In [ ]:
import numpy as np

def all_bitstrings(n_bits):
    values = np.arange(2 ** n_bits)
    return ((values[:, None] >> np.arange(n_bits)[::-1]) & 1)

def hamming_kernel(bitstrings, sigma):
    diff = bitstrings[:, None, :] != bitstrings[None, :, :]
    distances = diff.sum(axis=-1)
    return np.exp(-(distances) / (2.0 * sigma ** 2))

def mmd2(prob_p, prob_q, kernel):
    return (
        prob_p @ kernel @ prob_p
        + prob_q @ kernel @ prob_q
        - 2.0 * prob_p @ kernel @ prob_q
    )

def finite_difference_gradient(theta, objective, step=1e-3):
    grad = np.zeros_like(theta)
    for index in np.ndindex(theta.shape):
        shifted_plus = theta.copy()
        shifted_minus = theta.copy()
        shifted_plus[index] += step
        shifted_minus[index] -= step
        grad[index] = (
            objective(shifted_plus) - objective(shifted_minus)
        ) / (2.0 * step)
    return grad

def train_qcbm(theta, target_prob, circuit_probabilities,
               n_steps=200, learning_rate=0.05, sigma=1.0):
    bitstrings = all_bitstrings(int(np.log2(len(target_prob))))
    kernel = hamming_kernel(bitstrings, sigma=sigma)
    history = []

    def objective(current_theta):
        model_prob = circuit_probabilities(current_theta)
        return mmd2(target_prob, model_prob, kernel)

    for step in range(n_steps):
        loss = objective(theta)
        grad = finite_difference_gradient(theta, objective)
        theta = theta - learning_rate * grad
        history.append(loss)

    return theta, np.array(history)

In [ ]:
# Smoke test for the Chapter 10 framework-neutral training skeleton.
target_prob = np.array([0.80, 0.20], dtype=float)
theta0 = np.zeros_like(target_prob)

def circuit_probabilities(theta):
    weights = np.exp(theta - np.max(theta))
    return weights / weights.sum()

kernel = hamming_kernel(all_bitstrings(1), sigma=1.0)
initial_loss = mmd2(target_prob, circuit_probabilities(theta0), kernel)

trained_theta, history = train_qcbm(
    theta0,
    target_prob,
    circuit_probabilities,
    n_steps=40,
    learning_rate=0.8,
    sigma=1.0,
)
final_prob = circuit_probabilities(trained_theta)
final_loss = mmd2(target_prob, final_prob, kernel)

assert history.shape == (40,)
assert np.isclose(final_prob.sum(), 1.0)
assert np.isfinite(history).all()
assert final_loss < initial_loss

print({
    "initial_mmd2": round(float(initial_loss), 6),
    "final_mmd2": round(float(final_loss), 6),
})